[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qualia906/Developing-Agentic-AI-with-LangChain/blob/main/chap02/solution/problem/chap02_solution.ipynb)

# 第2章 演習（正解）: LangChain エージェントの基本構築

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap02-solution"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("環境変数の設定が完了しました")

## 課題1の正解: LLMを初期化する

In [ ]:
# ✅ 正解: init_chat_model をインポートして LLM を初期化
from langchain.chat_models import init_chat_model

# 「プロバイダー:モデル名」の形式で統一的に指定できる
llm = init_chat_model("openai:gpt-4o")

# LLM 単体でテスト
response = llm.invoke("こんにちは！一言で自己紹介してください。")
print("LLM 応答:", response.content)

## 課題2の正解: エージェントを作成する

In [ ]:
# ✅ 正解: create_agent をインポートしてエージェントを作成
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-4o",       # 使用するモデル名
    tools=[],                     # ツールなし（この章では LLM の会話能力のみ）
    system_prompt=(               # エージェントの役割を定義するシステムプロンプト
        "あなたは親切で知識豊富なAIアシスタントです。"
        "日本語で丁寧に回答してください。"
    ),
)

print("エージェントの作成が完了しました")

## 課題3の正解: エージェントに質問する

In [ ]:
# ✅ 正解: agent.invoke() に {"messages": [...]} を渡す
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "LangGraph とは何ですか？エージェント開発においてどんな役割を果たしますか？"
        }
    ]
})

# 最後のメッセージが AI エージェントの返答
print("エージェントの回答:")
print(result["messages"][-1].content)

## 確認の正解: メッセージ構造を見てみよう

In [ ]:
# ✅ 正解: ループでメッセージの型と内容を表示する
for i, msg in enumerate(result["messages"]):
    content_preview = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    print(f"[{i+1}] {type(msg).__name__}: {content_preview}")

## 【オプション】対話的な実行

`input()` 関数を使って、Google Colab 上で直接プロンプトを入力してエージェントを実行してみましょう。

In [ ]:
# Pythonの input() を使って対話的に質問を入力します
# 実行すると入力欄が表示されるので、質問を入力してEnterを押してください
interactive_input = input("エージェントへの入力: ")

In [ ]:
# 入力された質問でエージェントを実行します
result_interactive = agent.invoke({
    "messages": [
        {"role": "user", "content": interactive_input}
    ]
})

print("回答:")
print(result_interactive["messages"][-1].content)

---

## 課題4の正解: マルチモーダルコンテンツ


### 課題4-1の正解: URL で画像を指定して LLM に分析させる


In [ ]:
# ✅ 正解: URL 指定のコンテンツブロック形式
from langchain.chat_models import init_chat_model

mm_model = init_chat_model("openai:gpt-4o")

image_url_message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "この画像に何が写っていますか？日本語で説明してください。"},
        {
            "type": "image",
            "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png",
        },
    ],
}

response = mm_model.invoke([image_url_message])
print("=== URL 指定の画像入力結果 ===")
print(response.content)
print(f"\n入力トークン数: {response.usage_metadata['input_tokens']}")
print(f"出力トークン数: {response.usage_metadata['output_tokens']}")


### 課題4-2の正解: Base64 でローカル画像を埋め込む


In [ ]:
# ✅ 正解: サンプル画像をダウンロード
import urllib.request

sample_url = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/"
    "PNG_transparency_demonstration_1.png/"
    "280px-PNG_transparency_demonstration_1.png"
)
local_path = "sample_image.png"
urllib.request.urlretrieve(sample_url, local_path)
print(f"画像を {local_path} に保存しました")


In [ ]:
# ✅ 正解: Base64 エンコードしてコンテンツブロックで渡す
import base64

with open(local_path, "rb") as f:
    image_data = base64.standard_b64encode(f.read()).decode("utf-8")

image_base64_message = {
    "role": "user",
    "content": [
        {"type": "text", "text": "この画像の内容を詳しく説明してください。"},
        {
            "type": "image",
            "base64": image_data,
            "mime_type": "image/png",
        },
    ],
}

response_b64 = mm_model.invoke([image_base64_message])
print("=== Base64 埋め込みの画像入力結果 ===")
print(response_b64.content)


### 課題4-3の正解: エージェントに画像付きメッセージを渡す


In [ ]:
# ✅ 正解: create_agent に画像付きコンテンツブロックを渡す
from langchain.agents import create_agent

mm_agent = create_agent(
    model="openai:gpt-4o",
    tools=[],
    system_prompt="あなたは画像分析を専門とするAIアシスタントです。画像の内容を丁寧に日本語で説明してください。",
)

agent_result = mm_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "この画像を分析して、何が描かれているか教えてください。また、どのような場面で使われそうか考察してください。",
                },
                {
                    "type": "image",
                    "url": "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png",
                },
            ],
        }
    ]
})

print("=== エージェントによる画像分析 ===")
print(agent_result["messages"][-1].content)
